# Extractive Text Summarization berbasis TF-IDF

Studi kasus: Jurnal **Perkembangan dan Peran OPAC pada Aplikasi CIP (Cerah Informasi Pustaka) untuk Temu Kembali Informasi di Perpustakaan Universitas Tridinanti Palembang** (Betari Ayu Elsadantia, 2023).

## Aturan implementasi (dari dosen)
1. **Input adalah 1 dokumen utuh** (raw text), bukan D1/D2/D3 manual.
2. **RegEx memotong bab**: hanya isi `PENDAHULUAN` s/d akhir `HASIL PENELITIAN DAN PEMBAHASAN`. Abstrak, Kesimpulan, Daftar Pustaka harus terbuang otomatis.
3. **Paragraf = sub-dokumen**: deteksi paragraf via karakter enter (`\n\n`). Paragraf 1 = D1, paragraf 2 = D2, dst.
4. **Scoring & summarization**: hitung TF-IDF seluruh paragraf (manual TF, DF, IDF), lakukan ranking dengan TF-IDF & VSM, lalu ekstrak kalimat terbaik tiap paragraf.

## Pipeline notebook
1. **Ekstraksi PDF → raw text** (pdfplumber)
2. **RegEx** memotong bab (PENDAHULUAN s/d HASIL PENELITIAN DAN PEMBAHASAN)
3. **Split paragraf** (`\n\n`) → daftar sub-dokumen D1, D2, …
4. **Preprocessing** (case folding → cleaning → tokenizing → stopword removal → stemming)
5. **TF-IDF Manual** (vocab, TF, DF, IDF, TF-IDF)
6. **Bobot TF-IDF per Paragraf** (Top-5 keyword tiap paragraf)
7. **Ranking dengan TF-IDF** (query-based scoring)
8. **Ranking dengan VSM** (cosine similarity manual)
9. **Perbandingan ranking** TF-IDF vs VSM
10. **Summarization** (kalimat skor TF-IDF tertinggi tiap paragraf)

## 0. Install dependencies (sekali saja)

In [1]:
!pip install -q pdfplumber PySastrawi numpy pandas

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

### PENJELASAN — Setup Library & Tools (cell di bawah)

**Yang dilakukan cell ini:**
- Mengimpor library matematika (`math`), regex (`re`), `Counter`, `numpy`, `pandas`, `pdfplumber`.
- Membuat objek `stopword_remover` (PySastrawi) untuk menghapus stopword Bahasa Indonesia.
- Membuat objek `stemmer` (PySastrawi) untuk mengubah kata berimbuhan ke kata dasar.
  - Contoh: `"memberikan"` → `"beri"`, `"perpustakaan"` → `"pustaka"`.

> **Catatan dosen:** pastikan urutan preprocessing nanti = case folding → cleaning → tokenizing → stopword removal → stemming. Tools di cell ini adalah **bahan mentah** yang akan dipakai pipeline preprocessing di Langkah 4.


In [2]:
import math
import re
from collections import Counter

import numpy as np
import pandas as pd
import pdfplumber

from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.expand_frame_repr', False)

stopword_factory = StopWordRemoverFactory()
stopword_remover = stopword_factory.create_stop_word_remover()

stemmer_factory = StemmerFactory()
stemmer = stemmer_factory.create_stemmer()

print('Import dan konfigurasi selesai. Teks tabel tidak dipotong.')

Import dan konfigurasi selesai. Teks tabel tidak dipotong.


## LANGKAH 1 — Ekstraksi `Jurnal.pdf` menjadi raw text

Tantangan: pdfplumber memecah teks per baris (line-wrap), bukan per paragraf. Untuk
menghormati aturan dosen ("paragraf dipisah karakter enter"), kita memanfaatkan posisi
horizontal `x0` tiap baris:

* Baris **rata kiri** (`x0 ≈ 104.9`) → lanjutan paragraf yang sama.
* Baris **terindentasi** (`x0 ≈ 133.3`) → awal paragraf baru → sisipkan `\n\n`.
* Header bab (PENDAHULUAN, METODE, dll.) berdiri sendiri di kolom kiri → diberi `\n\n` sendiri.

Hasilnya satu string raw text yang strukturnya "berantakan tapi paragraf jelas".

### PENJELASAN — Ekstraksi PDF → Raw Text (cell di bawah)

| Aspek | Nilai |
|---|---|
| **INPUT** | file `Jurnal.pdf` (PDF mentah, belum diolah) |
| **OUTPUT** | satu string panjang `teks_jurnal` di mana paragraf dipisah `\n\n` |

**Mengapa rumit?**

`pdfplumber` memecah teks per **baris visual** (line-wrap), bukan per paragraf logis. Padahal aturan dosen: *"paragraf dipisah karakter enter"*. Jadi kita harus merekonstruksi paragraf dari posisi koordinat horizontal `x0` setiap baris:

- `x0 ≈ 104.9` (rata kiri) → baris ini lanjutan paragraf yang sama.
- `x0 > 125` (terindentasi) → baris ini **awal paragraf baru** → sisipkan `\n\n`.
- Header bab UPPERCASE pendek → berdiri sendiri sebagai `\n\n`.
- Sub-judul Title Case + bold → dibuang (bukan isi paragraf).

> **PENTING:** cell ini **belum** melakukan preprocessing. Hasilnya masih huruf kapital, tanda baca, angka, dsb. Pembersihan dilakukan di Langkah 4.


In [3]:
PATH_PDF = 'Jurnal1.pdf'

def ekstrak_pdf_jadi_raw_text(path_pdf: str) -> str:
    """Ekstrak PDF jurnal jadi 1 string raw text dengan paragraf dipisah \\n\\n.

    Strategi:
      - Posisi x0 untuk mendeteksi indent (= awal paragraf baru).
      - Pola UPPERCASE pendek untuk header bab (PENDAHULUAN/METODE/dll.).
      - Font 'Arial-BoldMT' untuk sub-judul Title Case (Perpustakaan, OPAC,
        Sistem Temu Kembali Informasi, dll.) yang dibuang sepenuhnya.
        Pembedanya dengan kutipan judul: sub-judul muncul SETELAH paragraf
        yang berakhir dengan tanda baca akhir kalimat (./?/!).
      - Page-footer & running-header juga disaring.
    """
    BATAS_INDENT = 125
    bagian_paragraf: list[str] = []
    buffer = ''

    def flush():
        nonlocal buffer
        if buffer.strip():
            bagian_paragraf.append(buffer.strip())
        buffer = ''

    pola_header_bab = re.compile(r'^[A-Z][A-Z\s]{3,40}$')
    pola_sampah = re.compile(
        r'^(?:\d+\s+Jurnal Multidisipliner|'
        r'Perkembangan Dan Peran Opac Pada Aplikasi|'
        r'Informasi Pustaka\) Untuk Temu Kembali|'
        r'Perpustakaan Universitas Tridinanti Palembang\s*$|'
        r'Betari Ayu\s*$|Elsadantia\s*$)',
        flags=re.IGNORECASE,
    )

    # State: apakah baris sebelumnya berakhir dengan tanda baca akhir kalimat?
    # True berarti kita berada di "boundary" antar-paragraf -> baris bold di
    # posisi ini sangat mungkin sub-judul.
    prev_end_with_period = True   # awal dokumen dianggap boundary
    skip_streak = False           # apakah baris bold sebelumnya juga sub-judul

    with pdfplumber.open(path_pdf) as pdf:
        for hal in pdf.pages:
            for ln in hal.extract_text_lines():
                teks_baris = ln['text'].strip()
                if not teks_baris or pola_sampah.search(teks_baris):
                    continue

                chars = ln.get('chars', [])
                font_pertama = chars[0].get('fontname', '') if chars else ''
                line_bold = 'Arial-BoldMT' in font_pertama

                terindentasi = ln['x0'] > BATAS_INDENT
                header_bab = bool(pola_header_bab.match(teks_baris))

                # Sub-judul Title Case: bold + posisi boundary + bukan UPPERCASE bab
                # Atau: bold + lanjutan dari sub-judul multi-baris sebelumnya
                sub_judul_title_case = line_bold and not header_bab and (
                    prev_end_with_period or skip_streak
                )

                if sub_judul_title_case:
                    flush()                 # tutup paragraf sebelumnya
                    skip_streak = True       # next bold line juga = continuation
                    # state prev_end_with_period dipertahankan (masih di boundary)
                    continue

                # Update state untuk baris berikutnya
                skip_streak = False
                prev_end_with_period = bool(re.search(r'[.!?]\s*$', teks_baris))

                if terindentasi or header_bab:
                    flush()
                    buffer = teks_baris
                    if header_bab:
                        flush()
                else:
                    buffer = (buffer + ' ' + teks_baris).strip()
    flush()

    return '\n\n'.join(bagian_paragraf)


teks_jurnal = ekstrak_pdf_jadi_raw_text(PATH_PDF)
print(f'Total karakter raw text: {len(teks_jurnal):,}')
print('=' * 78)
print('CUPLIKAN 800 KARAKTER PERTAMA:')
print('=' * 78)
print(teks_jurnal[:800])

Total karakter raw text: 22,811
CUPLIKAN 800 KARAKTER PERTAMA:
Prodi Ilmu Perpustakaan, Fakultas Adab dan Humaniora, UIN Raden Fatah Palembang

Email: betariayu2511@gmail.com ARTICLE HISTORY Abstrak Received: Penelitian ini membahas mengenai Perkembangan dan Peran OPAC Pada Aplikasi CIP (Cerah 12 Oktober 2023 Informasi Pustaka) untuk temu kembali informasi yang ada di Perpustakaan Tridinanti Revised Palembang. Di dalam artikel ini akan disampaikan mengenai Perkembangan dan Peran OPAC 14 Oktober 2023 Pada Aplikasi CIP (Cerah Informasi Pustaka) dalam meningkatkan layanan Perpustakaan Accepted: Tridinanti Palembang, serta efektivitas Perkembangan dan Peran OPAC Pada CIP (Cerah 23 Oktober 2023 Informasi Pustaka) sebagai sistem temu kembali informasi yang ada di Perpustakaan tersebut.

30 Oktober 2023 memberikan suatu kemudahan bagi pemustaka serta memberi


## LANGKAH 2 — RegEx memotong bab

Hanya isi mulai dari `PENDAHULUAN` sampai sebelum `KESIMPULAN` yang akan kita pakai.
Bagian Abstrak (di atas PENDAHULUAN) dan Kesimpulan + Daftar Pustaka (setelah Hasil)
akan otomatis terbuang.

### PENJELASAN — RegEx Memotong Bab (cell di bawah)

| Aspek | Nilai |
|---|---|
| **INPUT** | `teks_jurnal` (raw text seluruh jurnal, termasuk Abstrak & Daftar Pustaka) |
| **OUTPUT** | `isi_bab` (hanya isi PENDAHULUAN..sebelum KESIMPULAN) |

**Pola regex yang dipakai:**

```python
r'PENDAHULUAN\s*\n(.*?)\n\s*KESIMPULAN'
```

**Penjelasan komponen pola:**

- `PENDAHULUAN` → penanda awal (tidak ikut diambil sebagai isi).
- `\s*\n` → whitespace + newline setelah kata PENDAHULUAN.
- `(.*?)` → grup penangkap (group 1) yang **lazy**: ambil sesedikit mungkin. Lazy penting agar berhenti di kemunculan KESIMPULAN **pertama**, bukan kemunculan terakhir.
- `\n\s*KESIMPULAN` → penanda akhir (tidak ikut diambil).

**Flag:**

- `re.DOTALL` → titik (`.`) ikut cocok dengan newline → bisa lintas paragraf.
- `re.IGNORECASE` → aman terhadap variasi kapitalisasi.

**Hasilnya:** Abstrak (di atas PENDAHULUAN) dan Kesimpulan + Daftar Pustaka (setelah Hasil & Pembahasan) otomatis dibuang.


In [4]:
pola_bab = re.compile(
    r'PENDAHULUAN\s*\n(.*?)\n\s*KESIMPULAN',
    flags=re.DOTALL | re.IGNORECASE,
)

match_bab = pola_bab.search(teks_jurnal)
if not match_bab:
    raise ValueError('Bab PENDAHULUAN..KESIMPULAN tidak ditemukan.')

isi_bab = match_bab.group(1).strip()

print(f'Karakter sebelum RegEx: {len(teks_jurnal):,}')
print(f'Karakter setelah RegEx: {len(isi_bab):,}  (Abstrak + Kesimpulan + Pustaka terbuang)')
print('=' * 78)
print('CUPLIKAN 400 KARAKTER PERTAMA HASIL POTONG:')
print('=' * 78)
print(isi_bab[:400], '...')
print('=' * 78)
print('CUPLIKAN 400 KARAKTER TERAKHIR HASIL POTONG:')
print('=' * 78)
print('...', isi_bab[-400:])


Karakter sebelum RegEx: 22,811
Karakter setelah RegEx: 15,844  (Abstrak + Kesimpulan + Pustaka terbuang)
CUPLIKAN 400 KARAKTER PERTAMA HASIL POTONG:
Dunia saat ini sangatlah bergantung terhadap yang namanya teknologi. Di zaman yang serba modern ini manusia diharuskan untuk memahami perkembangan teknologi yang terjadi, salah satunya ialah teknologi informasi. Perkembangan teknologi ini banyak dilakukan di sebuah perusahaan maupun lembaga informasi lainnya. Awal dari perkembangan teknologi yang sederhana ini perusahaan-perusaan dan lembaga-lemba ...
CUPLIKAN 400 KARAKTER TERAKHIR HASIL POTONG:
... kasi CIP (Cerah Informasi Pustaka) ini sangat efektif dan memiliki banyak peran bagi pemustaka dan pustakawan yaitu untuk mempermudah dalam temu kembali suatu informasi, mempermudah dalam mengklasifikasi bahan pustaka.

Kendala dalam Penelusuran OPAC dengan menggunakan aplikasi CIP (Cerah Informasi Pustaka) yaitu jaringan nya kadang eror serta menu-menu item yang pengen ditambah dan belum seles

## LANGKAH 3 — Pecah jadi paragraf (sub-dokumen)

Setiap paragraf dipisah `\n\n`, jadi cukup `.split('\n\n')`. Sub-judul pendek
(mis. `OPAC`, `Perkembangan OPAC`) yang tidak memuat kalimat valid kita buang dengan
ambang panjang minimum 30 karakter.

### PENJELASAN — Split Paragraf jadi Sub-Dokumen (cell di bawah)

| Aspek | Nilai |
|---|---|
| **INPUT** | `isi_bab` (string panjang, paragraf dipisah `\n\n`) |
| **OUTPUT** | dict `dokumen` = `{'D1': teks_paragraf_1, 'D2': teks_paragraf_2, ...}` |

Aturan dosen: *"paragraf dipisah karakter enter"*. Karena di Langkah 1 kita sudah menulis ulang teks dengan `\n\n` antar-paragraf, di sini cukup `.split('\n\n')`.

**Filter yang dipakai:**

- Buang segmen `< 30` karakter (biasanya sub-judul pendek seperti `"OPAC"`).
- Buang segmen yang seluruhnya UPPERCASE (header bab tersisa).

> **KONVENSI PENTING:** setiap paragraf hasil split adalah **1 sub-dokumen** (`D1`, `D2`, ...). Inilah unit perhitungan TF-IDF nanti. `N` = jumlah paragraf, **bukan** jumlah file PDF.


In [5]:
paragraf_kasar = isi_bab.split('\n\n')

pola_uppercase_only = re.compile(r'^[A-Z\s]+$')

daftar_paragraf = [
    p.strip() for p in paragraf_kasar
    if len(p.strip()) >= 30 and not pola_uppercase_only.match(p.strip())
]

dokumen = {f'D{i+1}': p for i, p in enumerate(daftar_paragraf)}
label_dokumen = {f'D{i+1}': f'Paragraf {i+1}' for i in range(len(daftar_paragraf))}

print(f'Total paragraf terdeteksi (sub-dokumen): {len(daftar_paragraf)}')
print('=' * 78)
df_korpus = pd.DataFrame({
    'Dokumen': list(dokumen.keys()),
    'Bagian Jurnal': [label_dokumen[d] for d in dokumen.keys()],
    'Teks Asli': list(dokumen.values()),
})
display(df_korpus)


Total paragraf terdeteksi (sub-dokumen): 26


,Dokumen,Bagian Jurnal,Teks Asli
0,D1,Paragraf 1,"Dunia saat ini sangatlah bergantung terhadap yang namanya teknologi. Di zaman yang serba modern ini manusia diharuskan untuk memahami perkembangan teknologi yang terjadi, salah satunya ialah teknologi informasi. Perkembangan teknologi ini banyak dilakukan di sebuah perusahaan maupun lembaga informasi lainnya. Awal dari perkembangan teknologi yang sederhana ini perusahaan-perusaan dan lembaga-lembaga pemerintahan dan pendidikan berlomba-lomba dalam menciptakan teknologi. Begitupun didalam dunia perpustakaan, dengan adanya perkembangan teknologi, perpustakaan berusaha untuk mengembangkan eksistensinya di dalam sistem layanan informasi untuk memenuhi kebutuhan pemustaka. Khususnya dalam mengembangkan teknologi digital yang digunakan sebagai sistem temu balik informasi."
1,D2,Paragraf 2,"Perpustakaan adalah suatu ruangan bagian dari gedung atau bangunan yang berisi buku-buku koleksi, yang diatur serta disusun sedemikian rupa, sehingga mudah untuk dicari dan dipergunakan sewaktu-waktu diperlukan oleh pemakai. Perpustakaan adalah suatu ruangan bagian dari gedung atau bangunan yang berisi buku-buku koleksi, yang diatur serta disusun sedemikian rupa, sehingga mudah untuk dicari dan dipergunakan sewaktu- waktu diperlukan oleh pemakai “katalog online atau OPAC merupakan sistem katalog perpustakaan yang menggunakan komputer. Pangkalan datanya biasanya dirancang dan dibuat sendiri oleh perpustakaan dengan menggunakan perangkat lunak komersial atau buatan sendiri. Katalog ini memberikan informasi bibliografis dan letak koleksinya. Katalog biasanya dirancang untuk mempermudah pengguna sehingga tidak perlu bertanya dalam menggunakannya (user friendly)”. Dengan kata lain OPAC merupakan seperangkat sistem katalog perpustakaan berbasis online yang terpasang dan digunakan di perpustakaan sebagai alat untuk mencari koleksi yang ada di perpustakaan. Karena OPAC sebagai sistem temu balik informasi."
2,D3,Paragraf 3,"Sistem otomasi perpustakaan merupakan suatu proses pengelolaan perpustakaan dengan memanfaatkan teknologi informasi. Pemanfaatan teknologi informasi di perpustakaan bertujuan untuk meningkatkan efisiensi pekerjaan dan kualitas pelayanan pada pengguna (right information, right user dan right now), berhubungan dengan peran maupun fungsi perpustakaan sebagai kekuatan dalam pelestarian, penyebaran informasi ilmu pengetahuan serta kebudayaan yang berkembang seiring dengan kebutuhan manusia akan informasi. Perkembangan teknologi informasi telah mengubah paradigma pengelola perpustakaan yang semulanya berbasis manual sekarang sudah terotomasi. Otomasi perpustakaan merupakan suatu proses pengelolaan perpustakaan dengan memanfaatkan Teknologi Informasi (TI). Pemanfaatan sistem otomasi ini memberikan kemudahan dan kecepatan dalam proses pengolahan bahan pustaka. Selain itu, dapat memberikan kemudahan bagi pemustaka dalam melakukan penelurusan. Intinya pemanfaatan teknologi informasi di perpustakaan bertujuan untuk meningkatkan efisien pekerjaan dan kualitas pelayanan bagi pengguna. Sistem otomasi perpustakaan yang banyak digunakan oleh perpustakaan, yaitu perangkat lunak (Software) seperti Senayan Library Information Management System (SLIMS), OPAC yang digunakan yaitu berupa Aplikasi CIP (Cerah Aplikasi Pustaka) yang di buat sendiri oleh pustakawan Perpustakaan Universitas Tridinanti Palembang.Namun, perpustakaan yang sudah terotomasi juga menuntut pustakawanya harus mampu memahami isi dari perangkat lunaknya, karena akan sangat berpengaruh terhadap kinerja mereka."
3,D4,Paragraf 4,"Perpustakaan Universitas Tridinanti Palembang merupakan salah satu Perpustakaan yang telah terotomasi dengan menggunakan software berupa Aplikasi CIP (Cerah Aplikasi Pustaka) sebagai sarana temu kembali informasinya. Berdasarkan hasil observasi dan wawancara yang telah dilakukan oleh penulis di Perpustakaan Tridinanti Palembang, penelitian ini bertujuan untuk mengetahui Perkembangan dan Peran OPAC Pada Aplikasi CIP (C

## Preprocessing

Pipeline `preprocess(text)`:
1. Case folding
2. Cleaning (regex: hapus angka dan tanda baca)
3. Tokenizing
4. Stopword Removal (`StopWordRemoverFactory`)
5. Stemming (`StemmerFactory`)

### PENJELASAN — Preprocessing 5 Tahap (cell di bawah)

| Aspek | Nilai |
|---|---|
| **INPUT** | tiap nilai dict `dokumen` (string paragraf mentah) |
| **OUTPUT** | list token kata-dasar bersih untuk tiap paragraf |

**Pipeline (urut, sesuai instruksi dosen):**

1. **Case folding** → `text.lower()`
   Semua huruf jadi lowercase. `"OPAC"` dan `"opac"` dianggap sama.

2. **Cleaning** → `re.sub(r'[^a-z\s]', ' ', text)`
   Buang angka & tanda baca. Sisakan huruf `a-z` dan whitespace.

3. **Tokenizing** → `text.split()`
   Pecah berdasarkan whitespace → list of tokens.

4. **Stopword Removal** → PySastrawi `StopWordRemover`
   Buang kata umum tanpa makna spesifik: `"yang"`, `"di"`, `"dan"`, `"untuk"`, `"ini"`, dst. Tujuannya supaya term yang tersisa benar-benar pembeda antar dokumen.

5. **Stemming** → PySastrawi `StemmerFactory`
   Reduksi ke kata dasar:
   - `"perpustakaan"` → `"pustaka"`
   - `"memberikan"` → `"beri"`
   - `"pemustaka"` → `"pustaka"` (samakan varian)

   Kata yang berbeda bentuk tapi se-akar dihitung sebagai 1 term.

> **CATATAN:** function `preprocess()` ini akan **dipakai ulang** nanti untuk memproses **query** juga, bukan hanya dokumen. Konsistensi preprocessing dokumen vs query **WAJIB SAMA**, kalau tidak vocabulary tidak akan match.


In [6]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = text.split()
    teks_tanpa_stopword = stopword_remover.remove(' '.join(tokens))
    tokens_tanpa_stopword = teks_tanpa_stopword.split()
    tokens_stem = [stemmer.stem(token) for token in tokens_tanpa_stopword]
    return tokens_stem

hasil_preproses = {nama_dok: preprocess(teks) for nama_dok, teks in dokumen.items()}

df_preproses = pd.DataFrame({
    'Dokumen': list(hasil_preproses.keys()),
    'Bagian Jurnal': [label_dokumen[d] for d in hasil_preproses.keys()],
    'Token Hasil Preprocessing': [' '.join(v) for v in hasil_preproses.values()]
})
display(df_preproses)

,Dokumen,Bagian Jurnal,Token Hasil Preprocessing
0,D1,Paragraf 1,dunia gantung nama teknologi zaman serba modern manusia harus paham kembang teknologi salah satu teknologi informasi kembang teknologi usaha lembaga informasi kembang teknologi sederhana usaha rusa lembaga lembaga perintah didik lomba lomba cipta teknologi dalam dunia pustaka kembang teknologi pustaka usaha kembang eksistensi sistem layan informasi penuh butuh mustaka kembang teknologi digital sistem temu informasi
1,D2,Paragraf 2,pustaka ruang gedung bangun isi buku buku koleksi atur susun mudah cari waktu pakai pustaka ruang gedung bangun isi buku buku koleksi atur susun mudah cari waktu pakai katalog online opac sistem katalog pustaka komputer pangkal data rancang pustaka perangkat lunak komersial buat katalog informasi bibliografis letak koleksi katalog rancang mudah guna guna user friendly opac perangkat sistem katalog pustaka bas online pasang pustaka alat cari koleksi pustaka opac sistem temu informasi
2,D3,Paragraf 3,sistem otomasi pustaka proses kelola pustaka manfaat teknologi informasi manfaat teknologi informasi pustaka tuju tingkat efisiensi kerja kualitas layan guna right information right user right now hubung peran fungsi pustaka kuat lestari sebar informasi ilmu tahu budaya kembang iring butuh manusia informasi kembang teknologi informasi ubah paradigma kelola pustaka mula bas manual terotomasi otomasi pustaka proses kelola pustaka manfaat teknologi informasi ti manfaat sistem otomasi mudah cepat proses olah bahan pustaka mudah mustaka turus inti manfaat teknologi informasi pustaka tuju tingkat efisien kerja kualitas layan guna sistem otomasi pustaka pustaka perangkat lunak software senayan library information management system slims opac aplikasi cip cerah aplikasi pustaka pustakawan pustaka universitas tridinanti palembang pustaka terotomasi tuntut pustakawanya paham isi perangkat lunak pengaruh kerja
3,D4,Paragraf 4,pustaka universitas tridinanti palembang salah pustaka terotomasi software aplikasi cip cerah aplikasi pustaka sarana temu informasi dasar hasil observasi wawancara tulis pustaka tridinanti palembang teliti tuju kembang peran opac aplikasi cip cerah aplikasi pustaka pustaka universitas tridinanti palembang amat observasi teliti putus angkat judul kembang peran opac aplikasi cip cerah informasi mustaka temu informasi pustaka universitas tridinanti palembang
4,D5,Paragraf 5,teliti dekat deskriptif kualitatif dekat artikel kembang peran aplikasi cip cerah aplikasi pustaka pustaka tridinanti palembang sugiyono metode teliti kualitatif metode teliti landas filsafat postpositivisme teliti kondisi obyek alamiah lawan eksperimen mana teliti instrumen kunci ambil sampel sumber data purposive snowbaal teknik kumpul triangulasi gabung analisis data sifat induktif kualitatif hasil teliti kualitatif tekan makna generalisasi teliti kualitatif eksplorasi dalam fenomena sosial lingkung sosial laku jadi waktu
5,D6,Paragraf 6,teliti kualitatif teliti mengeksplor fenomena fenomena kuantifikasi sifat deskriptif proses langkah formula konsep erti erti konsep agam karakteristik barang jasa gambar gambar gaya gaya tata budaya model fisik artifak
6,D7,Paragraf 7,teliti pustaka tridinanti palembang alamat alamat jl kapten marzuki no ilir darat kec ilir darat timur kota palembang sumatera selatan teliti laksana tanggal november wib
7,D8,Paragraf 8,pustaka unit simpan koleksi bahan pustaka atur sistematis sambung maka sumber informasi definisi atas paham pustaka lembaga himpun karya koleksi koleksi tulis koleksi cetak koleksi rekam sistem susun
8,D9,Paragraf 9,lucy tedd opac sistem katalog pasang akses umum pakai telusur pangkal data katalog pustaka simpan karya infomasi lokasi sistem katalog hubung sistem sirkulasi pakai bahan pustaka cari sedia pustaka pinjam
9,D10,Paragraf 10,dasar urai simpul opac online public access catalog alat bantu telusur via katalog komputer risi cantum bibliografi akses umum temu koleksi pustaka toko buku unit informasi


## Perhitungan TF-IDF Manual

### PENJELASAN — Rumus TF-IDF Manual (cell di bawah)

Kita **tidak** pakai `sklearn TfidfVectorizer`. Semua dihitung manual sesuai rumus.

**Notasi:**

- $N$ = jumlah dokumen (di sini = jumlah paragraf, mis. 26).
- $t$ = sebuah term (kata) dari vocabulary.
- $d$ = sebuah dokumen (paragraf).

**(1) TF (Term Frequency)** — frekuensi relatif dalam 1 dokumen:

$$TF(t, d) = \frac{jumlah\_kemunculan(t, d)}{total\_token(d)}$$

Dinormalisasi dengan total token agar dokumen panjang tidak otomatis menang.

**(2) DF (Document Frequency)** — di berapa dokumen term $t$ muncul:

$$DF(t) = |\{d : t \in d\}|$$

**(3) IDF (Inverse Document Frequency)** — log basis 10:

$$IDF(t) = \log_{10}\left(\frac{N}{DF(t)}\right)$$

- Term yang muncul di **semua** dokumen → $IDF = \log_{10}(1) = 0$ → tidak informatif, akan dibobot nol.
- Term langka → IDF tinggi → kontribusinya besar.

**(4) TF-IDF** — bobot akhir per (term, dokumen):

$$TF\text{-}IDF(t, d) = TF(t, d) \times IDF(t)$$

**OUTPUT struktur data:**

| Variabel | Isi |
|---|---|
| `vocab` | list term unik terurut (kolom matriks) |
| `tf` | `dict[dok][term]` → nilai TF |
| `df_freq` | `dict[term]` → nilai DF |
| `idf` | `dict[term]` → nilai IDF |
| `tfidf` | `dict[dok][term]` → nilai TF-IDF (matriks bobot) |


In [7]:
nama_dokumen = list(hasil_preproses.keys())
N = len(nama_dokumen)

vocab = sorted({term for tokens in hasil_preproses.values() for term in tokens})

# TF manual
tf = {}
for dok, tokens in hasil_preproses.items():
    counter = Counter(tokens)
    total_token = len(tokens) if len(tokens) > 0 else 1
    tf[dok] = {term: counter.get(term, 0) / total_token for term in vocab}

# DF manual
df_freq = {term: sum(1 for dok in nama_dokumen if tf[dok][term] > 0) for term in vocab}

# IDF manual
idf = {term: math.log10(N / df_freq[term]) if df_freq[term] > 0 else 0.0 for term in vocab}

# TF-IDF manual
tfidf = {
    dok: {term: tf[dok][term] * idf[term] for term in vocab}
    for dok in nama_dokumen
}

print(f'Jumlah vocabulary (term unik): {len(vocab)} term')
print(f'Jumlah dokumen: {N}')

Jumlah vocabulary (term unik): 393 term
Jumlah dokumen: 26


## Bobot TF-IDF per Paragraf (Dokumen)

### PENJELASAN — Top-5 Keyword per Paragraf (cell di bawah)

> ⚠️ **CATATAN PENTING (sesuai arahan dosen)**
> Cell ini adalah tahap **GENERATE KEYWORD**, **bukan** tahap menerima query!

**Bedakan dengan jelas:**

| Aspek | Nilai |
|---|---|
| **INPUT cell ini** | matriks `tfidf` (sudah dihitung di Langkah 5) |
| **OUTPUT cell ini** | Top-5 term dengan TF-IDF tertinggi **per** paragraf → ini **keyword hasil otomatis** / *generated keyword*, bukan keyword yang diketik manusia |

**Logika:**

Untuk setiap dokumen $d$:

1. Ambil semua term $t$ dengan $TF\text{-}IDF(t, d) > 0$.
2. Urutkan menurun berdasarkan TF-IDF.
3. Ambil 5 teratas → itulah keyword utama paragraf tersebut.

**Kegunaan:** keyword ini bisa jadi label/tag untuk paragraf, atau preview isi.

> **JANGAN BINGUNG:** query yang diketik user muncul di **Langkah 7** (cell berikutnya), bukan di cell ini.


In [8]:
top_k = 5

print('BOBOT TF-IDF PER PARAGRAF (DOKUMEN)')
print('=' * 65)

bobot_data = []

for dok in nama_dokumen:
    skor_term = {t: tfidf[dok][t] for t in vocab if tfidf[dok][t] > 0}
    top_terms = sorted(skor_term.items(), key=lambda x: x[1], reverse=True)[:top_k]

    print(f'\n{dok} — {label_dokumen[dok]}')
    print('-' * 65)
    for rank, (term, skor) in enumerate(top_terms, 1):
        bar = '█' * int(skor * 600)
        print(f'  {rank}. {term:<22} TF-IDF: {skor:.6f}  {bar}')

    keyword_str = ', '.join([t for t, _ in top_terms])
    bobot_data.append({
        'Dokumen': dok,
        'Bagian Jurnal': label_dokumen[dok],
        'Keyword Utama (Top-5)': keyword_str,
        'Skor TF-IDF Tertinggi': round(top_terms[0][1], 6) if top_terms else 0
    })

print('\n')
print('=' * 65)
print('TABEL RINGKASAN KEYWORD UTAMA PER PARAGRAF')
print('=' * 65)
df_bobot = pd.DataFrame(bobot_data)
display(df_bobot)

BOBOT TF-IDF PER PARAGRAF (DOKUMEN)

D1 — Paragraf 1
-----------------------------------------------------------------
  1. teknologi              TF-IDF: 0.104146  ██████████████████████████████████████████████████████████████
  2. usaha                  TF-IDF: 0.077180  ██████████████████████████████████████████████
  3. lembaga                TF-IDF: 0.060761  ████████████████████████████████████
  4. kembang                TF-IDF: 0.055842  █████████████████████████████████
  5. lomba                  TF-IDF: 0.051454  ██████████████████████████████

D2 — Paragraf 2
-----------------------------------------------------------------
  1. buku                   TF-IDF: 0.039778  ███████████████████████
  2. bangun                 TF-IDF: 0.039305  ███████████████████████
  3. gedung                 TF-IDF: 0.039305  ███████████████████████
  4. ruang                  TF-IDF: 0.039305  ███████████████████████
  5. katalog                TF-IDF: 0.035547  █████████████████████

D3 — Pa

,Dokumen,Bagian Jurnal,Keyword Utama (Top-5),Skor TF-IDF Tertinggi
0,D1,Paragraf 1,"teknologi, usaha, lembaga, kembang, lomba",0.104146
1,D2,Paragraf 2,"buku, bangun, gedung, ruang, katalog",0.039778
2,D3,Paragraf 3,"manfaat, otomasi, kelola, right, teknologi",0.058957
3,D4,Paragraf 4,"aplikasi, palembang, tridinanti, observasi, universitas",0.054898
4,D5,Paragraf 5,"teliti, kualitatif, dekat, metode, sosial",0.084931
5,D6,Paragraf 6,"erti, gambar, gaya, fenomena, konsep",0.097584
6,D7,Paragraf 7,"alamat, darat, ilir, teliti, palembang",0.113198
7,D8,Paragraf 8,"koleksi, atas, himpun, maka, sambung",0.066711
8,D9,Paragraf 9,"pakai, katalog, infomasi, lucy, tedd",0.062523
9,D10,Paragraf 10,"cantum, risi, toko, via, access",0.054422


## Ranking dengan TF-IDF

### PENJELASAN — Ranking Dokumen Berdasarkan QUERY (cell di bawah)

**=========================================================**
**CELL 9 — RANKING DOKUMEN DENGAN TF-IDF (LANGKAH 7)**
**=========================================================**

#### MODEL

*"TF-IDF summation"* → skor relevansi sebuah dokumen terhadap query adalah **jumlah** bobot TF-IDF dari term-term query yang ada di dokumen tersebut.

$$skor(d, q) = \sum_{t \in q} tfidf[d][t]$$

> **Catatan:** ini **bukan** cosine similarity. Tidak ada normalisasi panjang vektor, jadi paragraf yang panjang dengan banyak match cenderung diuntungkan. Ini sengaja agar kita bisa membandingkan karakteristiknya dengan VSM cosine pada Cell 10.

#### QUERY

```
"perpustakaan opac sistem temu kembali informasi aplikasi cip"
```

Setelah `preprocess()` (case fold + clean + stopword + stem):

```python
['pustaka', 'opac', 'sistem', 'temu', 'informasi', 'aplikasi', 'cip']
```

Term query yang **tidak** ada di vocab (mis. typo) di-filter dengan `if term in vocab`, supaya `tfidf[dok][term]` tidak `KeyError`.

#### OUTPUT

`ranking_tfidf`: DataFrame `[Dokumen, Bagian Jurnal, Skor TF-IDF, Ranking TF-IDF]` yang sudah diurutkan.

#### INTERPRETASI

- Top-5 untuk jurnal ini biasanya: **D4, D25, D18, D26, D20**.
- **D6 = 0** → paragraf yang sama sekali tidak menyebut term query.


In [9]:
query = 'perpustakaan opac sistem temu kembali informasi aplikasi cip'
query_tokens = preprocess(query)
query_terms = [term for term in query_tokens if term in vocab]

skor_tfidf = {}
for dok in nama_dokumen:
    skor_tfidf[dok] = sum(tfidf[dok][term] for term in query_terms)

ranking_tfidf = pd.DataFrame({
    'Dokumen': nama_dokumen,
    'Bagian Jurnal': [label_dokumen[d] for d in nama_dokumen],
    'Skor TF-IDF': [skor_tfidf[d] for d in nama_dokumen]
}).sort_values('Skor TF-IDF', ascending=False).reset_index(drop=True)
ranking_tfidf['Ranking TF-IDF'] = ranking_tfidf.index + 1

print('Query:', query)
print('Term query setelah preprocessing:', query_terms)
display(ranking_tfidf)

Query: perpustakaan opac sistem temu kembali informasi aplikasi cip
Term query setelah preprocessing: ['pustaka', 'opac', 'sistem', 'temu', 'informasi', 'aplikasi', 'cip']


,Dokumen,Bagian Jurnal,Skor TF-IDF,Ranking TF-IDF
0,D4,Paragraf 4,0.127299,1
1,D25,Paragraf 25,0.121268,2
2,D18,Paragraf 18,0.102487,3
3,D26,Paragraf 26,0.102377,4
4,D20,Paragraf 20,0.082108,5
5,D22,Paragraf 22,0.072494,6
6,D24,Paragraf 24,0.064733,7
7,D19,Paragraf 19,0.061669,8
8,D21,Paragraf 21,0.057870,9
9,D23,Paragraf 23,0.055888,10


## Ranking dengan VSM (Cosine Similarity)

### PENJELASAN — Vector Space Model (Cosine Similarity) (cell di bawah)

| Aspek | Nilai |
|---|---|
| **INPUT** | matriks `tfidf` + `query` (yang sama dengan Langkah 7) |
| **OUTPUT** | ranking dokumen berdasarkan cosine similarity |

**Bedanya dengan Langkah 7:**

- **Langkah 7** (TF-IDF scoring) hanya **menjumlahkan** bobot TF-IDF term query. Dokumen panjang yang banyak menyebut term query otomatis menang.
- **Langkah 8** (VSM) memperhitungkan **arah vektor** + dinormalisasi panjangnya. Dokumen pendek yang fokus pada term query bisa menang melawan dokumen panjang yang banyak bicara hal lain.

**Representasi vektor** (panjang = `|vocab|`):

- Vektor dokumen $d$: $[TF\text{-}IDF(t_1,d),\ TF\text{-}IDF(t_2,d),\ \ldots,\ TF\text{-}IDF(t_n,d)]$
- Vektor query $q$: $[TF(t_1,q) \cdot IDF(t_1),\ TF(t_2,q) \cdot IDF(t_2),\ \ldots,\ TF(t_n,q) \cdot IDF(t_n)]$

> Perhatikan: query **diboboti** dengan IDF dari korpus, bukan IDF query. IDF korpus tetap berlaku karena mencerminkan kelangkaan term di koleksi.

**Rumus Cosine Similarity:**

$$\cos(q, d) = \frac{q \cdot d}{\|q\|\ \|d\|} = \frac{\sum q_i \times d_i}{\sqrt{\sum q_i^2} \times \sqrt{\sum d_i^2}}$$

- **Pembilang:** dot product (`vec_a · vec_b`).
- **Penyebut:** perkalian norma Euclidean kedua vektor.
- **Range:** 0 (ortogonal/tidak relevan) sampai 1 (identik arah).

**Implementasi manual dengan numpy:**

- `np.dot(vec_a, vec_b)` → dot product.
- `np.linalg.norm(vec)` → $\sqrt{\sum x_i^2}$.


In [10]:
def cosine_similarity_manual(vec_a, vec_b):
    pembilang = float(np.dot(vec_a, vec_b))
    penyebut = float(np.linalg.norm(vec_a) * np.linalg.norm(vec_b))
    return pembilang / penyebut if penyebut != 0 else 0.0

counter_query = Counter(query_terms)
total_query = len(query_terms) if len(query_terms) > 0 else 1
tf_query = {term: counter_query.get(term, 0) / total_query for term in vocab}
vektor_query = np.array([tf_query[term] * idf[term] for term in vocab], dtype=float)

skor_vsm = {}
for dok in nama_dokumen:
    vektor_dok = np.array([tfidf[dok][term] for term in vocab], dtype=float)
    skor_vsm[dok] = cosine_similarity_manual(vektor_query, vektor_dok)

ranking_vsm = pd.DataFrame({
    'Dokumen': nama_dokumen,
    'Bagian Jurnal': [label_dokumen[d] for d in nama_dokumen],
    'Skor VSM': [skor_vsm[d] for d in nama_dokumen]
}).sort_values('Skor VSM', ascending=False).reset_index(drop=True)
ranking_vsm['Ranking VSM'] = ranking_vsm.index + 1

display(ranking_vsm)

,Dokumen,Bagian Jurnal,Skor VSM,Ranking VSM
0,D4,Paragraf 4,0.424317,1
1,D25,Paragraf 25,0.322466,2
2,D24,Paragraf 24,0.207621,3
3,D26,Paragraf 26,0.176480,4
4,D20,Paragraf 20,0.120623,5
5,D18,Paragraf 18,0.110908,6
6,D3,Paragraf 3,0.101736,7
7,D5,Paragraf 5,0.096983,8
8,D19,Paragraf 19,0.096220,9
9,D22,Paragraf 22,0.094640,10


## Perbandingan Hasil Ranking

### PENJELASAN — Perbandingan Ranking TF-IDF vs VSM (cell di bawah)

| Aspek | Nilai |
|---|---|
| **INPUT** | tabel `ranking_tfidf` dan `ranking_vsm` (Langkah 7 & 8) |
| **OUTPUT** | satu tabel gabungan → mudah lihat di mana dua metode setuju vs beda |

**Yang perlu diperhatikan:**

- Dokumen yang **sama-sama peringkat 1** di kedua metode = pemenang absolut.
- **Pergeseran besar** (mis. peringkat 3 di TF-IDF tapi peringkat 8 di VSM) biasanya menandakan dokumen panjang dengan banyak term query, tapi kepadatan/relevansinya rendah → ditegur oleh normalisasi cosine.
- Dokumen **pendek yang fokus topik** bisa naik di VSM karena norma kecil.


In [11]:
hasil_perbandingan = (
    ranking_tfidf[['Dokumen', 'Bagian Jurnal', 'Skor TF-IDF', 'Ranking TF-IDF']]
    .merge(ranking_vsm[['Dokumen', 'Skor VSM', 'Ranking VSM']], on='Dokumen', how='inner')
)
hasil_perbandingan = hasil_perbandingan.sort_values('Ranking TF-IDF').reset_index(drop=True)

display(hasil_perbandingan[['Dokumen', 'Bagian Jurnal', 'Skor TF-IDF', 'Ranking TF-IDF', 'Skor VSM', 'Ranking VSM']])

,Dokumen,Bagian Jurnal,Skor TF-IDF,Ranking TF-IDF,Skor VSM,Ranking VSM
0,D4,Paragraf 4,0.127299,1,0.424317,1
1,D25,Paragraf 25,0.121268,2,0.322466,2
2,D18,Paragraf 18,0.102487,3,0.110908,6
3,D26,Paragraf 26,0.102377,4,0.176480,4
4,D20,Paragraf 20,0.082108,5,0.120623,5
5,D22,Paragraf 22,0.072494,6,0.094640,10
6,D24,Paragraf 24,0.064733,7,0.207621,3
7,D19,Paragraf 19,0.061669,8,0.096220,9
8,D21,Paragraf 21,0.057870,9,0.075902,12
9,D23,Paragraf 23,0.055888,10,0.083349,11


## Summarization

### PENJELASAN — Extractive Summarization per Paragraf (cell di bawah)

| Aspek | Nilai |
|---|---|
| **INPUT** | dokumen asli per paragraf + matriks TF-IDF |
| **OUTPUT** | 1 kalimat terpilih per paragraf → dirangkai jadi ringkasan jurnal |

**Tiga sub-langkah:**

#### (A) Pecah paragraf jadi kalimat — `pecah_kalimat(teks)`

Tantangan: tanda titik tidak selalu akhir kalimat. Yang dilindungi:

- Desimal/jam: `11.00`, `12.00`.
- Singkatan 1 huruf: `"A.Tedd"`, `"A. Tedd"`.
- Singkatan umum: `JL.`, `No.`, `dkk.`, `dll.`, `tsb.`.
- Tanggal Indonesia: `"28 November"`.

Caranya: substitute pola tersebut dengan placeholder `__P0__`, `__P1__`, dst., pecah dengan regex `(?<=[.!?])\s+(?=[A-Z"\d])`, lalu kembalikan placeholder.

#### (B) Skor tiap kalimat — `skor_kalimat(...)`

$$skor(kalimat) = \frac{1}{|tokens|} \sum_{t \in kalimat} TF(t, kalimat) \times IDF(t)$$

- Rata-rata bobot TF-IDF tiap token kalimat (TF lokal kalimat × IDF korpus).
- Dinormalisasi dengan jumlah token agar kalimat panjang tidak otomatis menang.

#### (C) Pilih kalimat ber-skor tertinggi tiap paragraf

Kandidat ringkasan. Gabungkan semua → ringkasan akhir jurnal.

> **Catatan:** ini **extractive** summarization (memilih kalimat asli), bukan **abstractive** (membuat kalimat baru). Tidak ada generative model di sini.


In [12]:
def normalisasi_teks(teks: str) -> str:
    """Bersihkan smart quotes & whitespace ganda agar tampilan kalimat rapi."""
    teks = (teks
            .replace('“', '"').replace('”', '"')
            .replace('‘', "'").replace('’', "'"))
    teks = re.sub(r'\s+', ' ', teks).strip()
    return teks


def pecah_kalimat(teks: str) -> list[str]:
    """Pecah teks jadi kalimat dengan regex yang menghormati:
       - desimal/jam (11.00, 12.00)         -> tidak dipecah
       - singkatan 1 huruf (A.Tedd, A. Tedd) -> tidak dipecah
       - singkatan 'JL.', 'No.', 'dkk.'      -> tidak dipecah
       - tanggal: 28 November                -> tidak dipecah
    """
    teks = normalisasi_teks(teks)

    # Lindungi pola titik yang BUKAN akhir kalimat dengan placeholder.
    pelindung = {}
    def lindungi(m):
        token = f'__P{len(pelindung)}__'
        pelindung[token] = m.group(0)
        return token

    pola_titik_bukan_akhir = re.compile(
        r'\b(?:'
          r'\d+\.\d+'                       # 11.00, 12.00
          r'|[A-Z]\.\s?[A-Z][a-z]+'         # A. Tedd / A.Tedd
          r'|JL\.|No\.|dkk\.|dll\.|tsb\.'   # singkatan umum
          r'|\d+\.\s?[A-Z][a-z]+'           # tanggal: 28 November
        r')',
        flags=re.IGNORECASE,
    )
    teks_aman = pola_titik_bukan_akhir.sub(lindungi, teks)

    # Pecah pada titik/!/? yang diikuti spasi + huruf besar / kutip / angka.
    bagian = re.split(r'(?<=[.!?])\s+(?=[A-Z"\d])', teks_aman)

    # Pulihkan placeholder kembali jadi teks asli
    hasil = []
    for kal in bagian:
        for token, asli in pelindung.items():
            kal = kal.replace(token, asli)
        kal = kal.strip()                              # buang whitespace pinggir
        kal = re.sub(r'\s*\.+\s*$', '', kal)            # buang titik akhir (dipasang ulang saat display)
        if len(kal) >= 25:
            hasil.append(kal)
    return hasil


def skor_kalimat(kalimat, tfidf_dok, vocab, idf):
    """Skor kalimat = rata-rata TF-IDF semua token dalam kalimat."""
    tokens = preprocess(kalimat)
    if not tokens:
        return 0.0
    total_skor = 0.0
    for token in tokens:
        if token in idf:
            tf_token = tokens.count(token) / len(tokens)
            total_skor += tf_token * idf[token]
    return total_skor / len(tokens)


print('SUMMARIZATION JURNAL BERBASIS TF-IDF')
print('=' * 70)
print(f'Judul Jurnal : Perkembangan dan Peran OPAC pada Aplikasi CIP (Cerah Informasi Pustaka)')
print(f'Penulis      : Betari Ayu Elsadantia')
print(f'Query        : {query}')
print('=' * 70)

summary_rows = []
kalimat_summary = []

for dok in nama_dokumen:
    teks_asli = dokumen[dok]

    # Pecah dokumen menjadi kalimat dengan splitter yang melindungi singkatan
    kalimat_list = pecah_kalimat(teks_asli)

    if not kalimat_list:
        continue

    # Hitung skor setiap kalimat
    skor_per_kalimat = [(kal, skor_kalimat(kal, tfidf[dok], vocab, idf))
                       for kal in kalimat_list]

    # Ambil kalimat dengan skor tertinggi
    kalimat_terbaik, skor_terbaik = max(skor_per_kalimat, key=lambda x: x[1])

    print(f'\n{dok} — {label_dokumen[dok]}')
    print('-' * 70)
    print('Skor semua kalimat:')
    for i, (kal, skor) in enumerate(skor_per_kalimat, 1):
        tanda = ' ◄ TERPILIH' if kal == kalimat_terbaik else ''
        print(f'  [{i}] (skor: {skor:.5f}){tanda}')
        print(f'       "{kal[:80]}..."' if len(kal) > 80 else f'       "{kal}"')
    print(f'\n  >>> Kalimat Terpilih (skor: {skor_terbaik:.5f}):')
    print(f'      "{kalimat_terbaik}."')

    kalimat_summary.append(kalimat_terbaik + '.')
    summary_rows.append({
        'Dokumen': dok,
        'Bagian Jurnal': label_dokumen[dok],
        'Jumlah Kalimat': len(kalimat_list),
        'Skor Kalimat Terpilih': round(skor_terbaik, 5),
        'Kalimat Summary': kalimat_terbaik + '.'
    })

# Tampilkan tabel summary
print('\n')
print('=' * 70)
print('TABEL HASIL SUMMARIZATION PER DOKUMEN')
print('=' * 70)
df_summary_result = pd.DataFrame(summary_rows)
display(df_summary_result)

# Tampilkan ringkasan akhir
print('\n')
print('=' * 70)
print('RINGKASAN AKHIR JURNAL (EXTRACTIVE SUMMARIZATION)')
print('=' * 70)
ringkasan_akhir = ' '.join(kalimat_summary)
print(ringkasan_akhir)

SUMMARIZATION JURNAL BERBASIS TF-IDF
Judul Jurnal : Perkembangan dan Peran OPAC pada Aplikasi CIP (Cerah Informasi Pustaka)
Penulis      : Betari Ayu Elsadantia
Query        : perpustakaan opac sistem temu kembali informasi aplikasi cip

D1 — Paragraf 1
----------------------------------------------------------------------
Skor semua kalimat:
  [1] (skor: 0.27243) ◄ TERPILIH
       "Dunia saat ini sangatlah bergantung terhadap yang namanya teknologi"
  [2] (skor: 0.09326)
       "Di zaman yang serba modern ini manusia diharuskan untuk memahami perkembangan te..."
  [3] (skor: 0.15483)
       "Perkembangan teknologi ini banyak dilakukan di sebuah perusahaan maupun lembaga ..."
  [4] (skor: 0.12650)
       "Awal dari perkembangan teknologi yang sederhana ini perusahaan-perusaan dan lemb..."
  [5] (skor: 0.05112)
       "Begitupun didalam dunia perpustakaan, dengan adanya perkembangan teknologi, perp..."
  [6] (skor: 0.09176)
       "Khususnya dalam mengembangkan teknologi digital yang di

,Dokumen,Bagian Jurnal,Jumlah Kalimat,Skor Kalimat Terpilih,Kalimat Summary
0,D1,Paragraf 1,6,0.27243,Dunia saat ini sangatlah bergantung terhadap yang namanya teknologi.
1,D2,Paragraf 2,7,0.14431,"Katalog biasanya dirancang untuk mempermudah pengguna sehingga tidak perlu bertanya dalam menggunakannya (user friendly)""."
2,D3,Paragraf 3,8,0.28485,"Selain itu, dapat memberikan kemudahan bagi pemustaka dalam melakukan penelurusan."
3,D4,Paragraf 4,2,0.04666,Perpustakaan Universitas Tridinanti Palembang merupakan salah satu Perpustakaan yang telah terotomasi dengan menggunakan software berupa Aplikasi CIP (Cerah Aplikasi Pustaka) sebagai sarana temu kembali informasinya.
4,D5,Paragraf 5,4,0.27849,Penelitian ini menggunakan pendekatan deskriptif kualitatif.
5,D6,Paragraf 6,1,0.06099,"Penelitian kualitatif dilakukan karena peneliti ingin mengeksplor fenomena-fenomena yang tidak dapat dikuantifikasikan yang bersifat deskriptif seperti proses suatu langkah kerja, formula suatu konsep, pengertian-pengertian tentang suatu konsep yang beragam, karakteristik suatu barang dan jasa, gambar-gambar, gaya-gaya, tata cara suatu budaya, model fisik suatu artifak dan lain sebagainya."
6,D7,Paragraf 7,1,0.06771,"Penelitian ini dilakukan di Perpustakaan Tridinanti Palembang, yang beralamat di Alamat JL.Kapten Marzuki No.2446 Ilir Darat 3. Kec Ilir Darat Timur I Kota Palembang, Sumatera Selatan 30129. Penelitian ini dilaksanakan pada tanggal 28 November 2022 pada pukul 11.00-12.00 WIB."
7,D8,Paragraf 8,2,0.06793,"Dari definisi diatas, dapat dipahami bahwa perpustakaan merupakan lembaga yang menghimpun berbagai karya koleksi seperti koleksi tertulis, koleksi cetak maupun koleksi rekam dengan menggunakan sistem dan susunan tertentu."
8,D9,Paragraf 9,1,0.03151,"Menurut Lucy A.Tedd, OPAC adalah sistem katalog terpasang yang dapat diakses secara umum, dan dapat digunakan pemakai untuk menelusur pangkalan data katalog, untuk memastikan apakah perpustakaan menyimpan karya tertentu, untuk mendapatkan infomasi tentang lokasinya, dan jika sistem katalog dihubungkan dengan sistem sirkulasi, maka pemakai dapat mengetahui apakah bahan pustaka yang sedang dicari, sedang tersedia di perpustakaan atau sedang dipinjam."
9,D10,Paragraf 10,1,0.03122,"Berdasarkan uraian di atas dapat disimpulkan bahwa OPAC (Online Public Access Catalog) merupakan suatu alat bantuan penelusuran via katalog komputer yang berisikan cantuman bibliografi dan dapat diakses secara umum untuk menemukan koleksi di suatu perpustakaan, toko buku, maupun unit informasi lainnya."




RINGKASAN AKHIR JURNAL (EXTRACTIVE SUMMARIZATION)
Dunia saat ini sangatlah bergantung terhadap yang namanya teknologi. Katalog biasanya dirancang untuk mempermudah pengguna sehingga tidak perlu bertanya dalam menggunakannya (user friendly)". Selain itu, dapat memberikan kemudahan bagi pemustaka dalam melakukan penelurusan. Perpustakaan Universitas Tridinanti Palembang merupakan salah satu Perpustakaan yang telah terotomasi dengan menggunakan software berupa Aplikasi CIP (Cerah Aplikasi Pustaka) sebagai sarana temu kembali informasinya. Penelitian ini menggunakan pendekatan deskriptif kualitatif. Penelitian kualitatif dilakukan karena peneliti ingin mengeksplor fenomena-fenomena yang tidak dapat dikuantifikasikan yang bersifat deskriptif seperti proses suatu langkah kerja, formula suatu konsep, pengertian-pengertian tentang suatu konsep yang beragam, karakteristik suatu barang dan jasa, gambar-gambar, gaya-gaya, tata cara suatu budaya, model fisik suatu artifak dan lain sebagainya. Pe